# Ejercicio · Point Transformer sobre Teeth3DS+ — segmentación de dientes (FDI)

Reproducción del ejemplo de entrenamiento de una **librería PyTorch para nubes de puntos**,
midiendo el **tiempo de entrenamiento** y entendiendo **los datos** (cuántos hay, qué es test).

- **Librería:** [PyTorch Geometric (PyG)](https://pytorch-geometric.readthedocs.io) `2.8` + `pyg-lib`
- **Modelo:** Point Transformer (segmentación) — [ejemplo oficial](https://github.com/pyg-team/pytorch_geometric/blob/master/examples/point_transformer_segmentation.py)
- **Datos:** **Teeth3DS+ local** (`data/raw/teeth3ds`) en vez de ShapeNet — misma tarea
  (segmentar partes de una nube), pero las "partes" son **dientes (FDI)**. Cero descarga.
- **Kernel:** `Dental GPU (3DGS)` (torch cu128 + RTX 5070)

> Es un **prototipo directo del `segmentation-agent`**: segmentar la nube de puntos por diente FDI.
> Reproduce el modelo/bucle oficial; solo cambia el *loader* (mallas `.obj` + labels `.json` → PyG)
> e IoU calculado a mano (la firma de `torchmetrics.jaccard_index` cambió en 1.x).

### 0 · Entorno

In [1]:
import torch, torch_geometric, pyg_lib
print("torch          :", torch.__version__)
print("torch_geometric:", torch_geometric.__version__, "| pyg_lib:", pyg_lib.__version__)
print("CUDA           :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

torch          : 2.11.0+cu128
torch_geometric: 2.8.0 | pyg_lib: 0.7.0+pt211cu128
CUDA           : True | NVIDIA GeForce RTX 5070


### 1 · Datos — Teeth3DS+ local (¿qué cantidad tenemos y qué es test?)

Cada caso = una arcada: malla `.obj` (~76k–168k vértices con color por vértice) + `.json` con
una etiqueta **FDI por vértice** (`0` = encía). Submuestreamos a 2048 puntos/muestra y hacemos
split **por paciente** (sin fuga: un paciente no está a la vez en train y test).

In [2]:
import glob, json, os, os.path as osp, time
from pathlib import Path
import numpy as np
import vtk
from vtk.util.numpy_support import vtk_to_numpy
import torch_geometric.transforms as T
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

vtk.vtkObject.GlobalWarningDisplayOff()  # los .obj traen color por vértice -> flood de avisos

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data/raw/teeth3ds").exists():
    ROOT = ROOT.parent
MESH = ROOT / "data/raw/teeth3ds/3D_scans_per_patient_obj_files"
LAB = ROOT / "data/raw/teeth3ds/ground-truth_labels_instances"
NUM_POINTS, BATCH = 2048, 4


def read_case(obj_path, json_path):
    """Devuelve pos (N,3), labels FDI (N,) y NORMALES por vértice (N,3).
    Normales SIN split (SetSplitting=0) -> mismo conteo/orden que pos y labels."""
    r = vtk.vtkOBJReader(); r.SetFileName(str(obj_path)); r.Update()
    poly = r.GetOutput()
    pos = vtk_to_numpy(poly.GetPoints().GetData()).astype(np.float32)
    nf = vtk.vtkPolyDataNormals()
    nf.SetInputData(poly); nf.SetSplitting(0); nf.SetConsistency(1)
    nf.SetComputePointNormals(1); nf.SetComputeCellNormals(0); nf.Update()
    nrm = vtk_to_numpy(nf.GetOutput().GetPointData().GetNormals()).astype(np.float32)
    lab = np.asarray(json.load(open(json_path))["labels"], dtype=np.int64)
    assert len(lab) == pos.shape[0] == nrm.shape[0]
    return pos, lab, nrm


# --- Gris CBCT SINTÉTICO (swap-ready) · comentario #2 del jefe ----------------
# PUNTO DE SWAP: hoy no hay CBCT emparejado, así que fabricamos un volumen de
# "atenuación" desde la propia malla (ocupación voxelizada + blur) y lo
# muestreamos por vértice. La MAQUINARIA (volumen 3D + muestreo trilineal en las
# posiciones de los vértices) es idéntica a la del CBCT real: cuando llegue el
# dato, se reemplaza `_synth_volume(...)` por cargar el volumen CBCT
# co-registrado y el resto no cambia. HONESTO: al derivarse de la superficie,
# este gris es un proxy de grosor/densidad local (correlaciona algo con el
# tejido), NO densidad interna real -> su validez definitiva espera al swap.
def _synth_volume(p, res=64, pad=0.05):
    lo, hi = p.min(0), p.max(0); d = hi - lo; lo = lo - pad * d; hi = hi + pad * d
    spacing = (hi - lo) / (res - 1)
    idx = np.floor((p - lo) / spacing).astype(int).clip(0, res - 1)
    vol = np.zeros((res, res, res), np.float32)
    np.add.at(vol, (idx[:, 0], idx[:, 1], idx[:, 2]), 1.0)
    k = np.array([1, 2, 1], np.float32); k /= k.sum()          # blur separable (falloff)
    for _ in range(3):
        for ax in range(3):
            vol = np.apply_along_axis(lambda m: np.convolve(m, k, mode="same"), ax, vol)
    return vol / max(vol.max(), 1e-8), lo.astype(np.float32), spacing.astype(np.float32)


def cbct_gray_synthetic(pos):
    """Muestreo trilineal del volumen sintético en cada vértice -> gris (N,)."""
    vol, lo, spacing = _synth_volume(pos)
    res = vol.shape[0]; g = (pos - lo) / spacing
    g0 = np.floor(g).astype(int); f = g - g0; out = np.zeros(len(pos), np.float32)
    for dx in (0, 1):
        for dy in (0, 1):
            for dz in (0, 1):
                gi = (g0 + [dx, dy, dz]).clip(0, res - 1)
                w = (np.where(dx, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1])
                     * np.where(dz, f[:, 2], 1 - f[:, 2]))
                out += w * vol[gi[:, 0], gi[:, 1], gi[:, 2]]
    return out


t0 = time.perf_counter()
patients = sorted(os.listdir(MESH))
raw = []
for pid in patients:
    for jaw in ("lower", "upper"):
        op, jp = MESH / pid / f"{pid}_{jaw}.obj", LAB / pid / f"{pid}_{jaw}.json"
        if op.exists() and jp.exists():
            raw.append((pid, *read_case(op, jp)))

all_codes = sorted({int(x) for _, _, lab, _ in raw for x in np.unique(lab)})
code2idx = {c: i for i, c in enumerate(all_codes)}   # 0(encía)->0, FDI->1..K-1
K = len(all_codes)
verts = [p.shape[0] for _, p, _, _ in raw]
print(f"{len(raw)} mallas de {len(patients)} pacientes en {time.perf_counter()-t0:.1f}s (con normales)")
print(f"clases (encía+FDI): {K}  ->  {all_codes}")
print(f"vértices/malla: min={min(verts)} media={sum(verts)//len(verts)} max={max(verts)} -> submuestreo a {NUM_POINTS}")

600 mallas de 300 pacientes en 75.7s (con normales)
clases (encía+FDI): 32  ->  [0, 11, 12, 13, 14, 15, 16, 17, 21, 22, 23, 24, 25, 26, 27, 28, 31, 32, 33, 34, 35, 36, 37, 38, 41, 42, 43, 44, 45, 46, 47, 48]
vértices/malla: min=13034 media=116527 max=219518 -> submuestreo a 2048


In [3]:
norm = T.NormalizeScale()
data_all = []
for pid, pos, lab, nrm in raw:
    y = torch.tensor([code2idx[int(x)] for x in lab], dtype=torch.long)
    gray = torch.from_numpy(cbct_gray_synthetic(pos))                 # <- SWAP: CBCT real
    feats = torch.cat([torch.from_numpy(nrm), gray[:, None]], dim=1)  # (N,4): nx ny nz gris
    d = norm(Data(pos=torch.from_numpy(pos), y=y))                    # norm solo escala pos
    d.x = feats; d.pid = pid
    data_all.append(d)
raw.clear()  # libera labels/normales crudos + tuplas (pos y x ya viven en data_all)

# Split REAL 80/20 POR PACIENTE. Semilla fija = reproducible; sin fuga (un
# paciente entero cae en train O en test). Con Teeth3DS+ completo: ~480/120 mallas.
ids = sorted({d.pid for d in data_all})
rng = np.random.default_rng(42); rng.shuffle(ids)
n_test = max(1, round(0.20 * len(ids)))
test_patients = set(ids[:n_test])
train_list = [d for d in data_all if d.pid not in test_patients]
test_list = [d for d in data_all if d.pid in test_patients]

# Estandariza el canal de gris con estadísticos de TRAIN (las normales ya son
# unitarias, se dejan tal cual). Sin fuga: media/std solo del train.
gtr = torch.cat([d.x[:, 3] for d in train_list])
gm, gs = gtr.mean(), gtr.std().clamp_min(1e-6)
for d in data_all:
    d.x[:, 3] = (d.x[:, 3] - gm) / gs

print(f"pacientes: {len(ids)}  ->  train {len(ids) - n_test} / test {n_test}  (80/20, seed 42)")
print(f"train = {len(train_list)} mallas / test = {len(test_list)} mallas")
print(f"features/punto: 4 (normal xyz + gris CBCT sintético) · gris train media {gm:.3f} std {gs:.3f}")

sampler = T.FixedPoints(NUM_POINTS, replace=False, allow_duplicates=True)


class TeethDS:
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return sampler(self.items[i].clone())  # submuestreo aleatorio/época
    num_classes = K


train_loader = DataLoader(TeethDS(train_list), batch_size=BATCH, shuffle=True)
test_loader = DataLoader(TeethDS(test_list), batch_size=BATCH, shuffle=False)
print("num_classes:", K)

pacientes: 300  ->  train 240 / test 60  (80/20, seed 42)
train = 480 mallas / test = 120 mallas
features/punto: 4 (normal xyz + gris CBCT sintético) · gris train media 0.436 std 0.171
num_classes: 32


### 2 · Modelo — Point Transformer (segmentación, *pos-only*)

In [4]:
import torch
import torch.nn.functional as F
from torch.nn import Linear as Lin
from torch_geometric.nn import MLP, PointTransformerConv, fps, knn, knn_graph, knn_interpolate
from torch_geometric.utils import scatter


class TransformerBlock(torch.nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.lin_in = Lin(ic, ic); self.lin_out = Lin(oc, oc)
        self.pos_nn = MLP([3, 64, oc], norm=None, plain_last=False)
        self.attn_nn = MLP([oc, 64, oc], norm=None, plain_last=False)
        self.transformer = PointTransformerConv(ic, oc, pos_nn=self.pos_nn, attn_nn=self.attn_nn)

    def forward(self, x, pos, ei):
        x = self.lin_in(x).relu(); x = self.transformer(x, pos, ei); return self.lin_out(x).relu()


class TransitionDown(torch.nn.Module):
    """Submuestrea (FPS) y agrupa features por kNN (max-pool)."""
    def __init__(self, ic, oc, ratio=0.25, k=16):
        super().__init__()
        self.k, self.ratio = k, ratio; self.mlp = MLP([ic, oc], plain_last=False)

    def forward(self, x, pos, batch):
        idc = fps(pos, ratio=self.ratio, batch=batch)
        sb = batch[idc] if batch is not None else None
        idk = knn(pos, pos[idc], k=self.k, batch_x=batch, batch_y=sb)
        x = self.mlp(x)
        xo = scatter(x[idk[1]], idk[0], dim=0, dim_size=idc.size(0), reduce="max")
        return xo, pos[idc], sb


class TransitionUp(torch.nn.Module):
    """Interpola features de baja resolución de vuelta a alta."""
    def __init__(self, ic, oc):
        super().__init__()
        self.mlp_sub = MLP([ic, oc], plain_last=False); self.mlp = MLP([oc, oc], plain_last=False)

    def forward(self, x, x_sub, pos, pos_sub, batch=None, batch_sub=None):
        x_sub = self.mlp_sub(x_sub)
        xi = knn_interpolate(x_sub, pos_sub, pos, k=3, batch_x=batch_sub, batch_y=batch)
        return self.mlp(x) + xi


class Net(torch.nn.Module):
    def __init__(self, ic, oc, dim_model, k=16):
        super().__init__()
        self.k = k; ic = max(ic, 1)
        self.mlp_input = MLP([ic, dim_model[0]], plain_last=False)
        self.transformer_input = TransformerBlock(dim_model[0], dim_model[0])
        self.transformers_up = torch.nn.ModuleList(); self.transformers_down = torch.nn.ModuleList()
        self.transition_up = torch.nn.ModuleList(); self.transition_down = torch.nn.ModuleList()
        for i in range(len(dim_model) - 1):
            self.transition_down.append(TransitionDown(dim_model[i], dim_model[i + 1], k=k))
            self.transformers_down.append(TransformerBlock(dim_model[i + 1], dim_model[i + 1]))
            self.transition_up.append(TransitionUp(dim_model[i + 1], dim_model[i]))
            self.transformers_up.append(TransformerBlock(dim_model[i], dim_model[i]))
        self.mlp_summit = MLP([dim_model[-1], dim_model[-1]], norm=None, plain_last=False)
        self.transformer_summit = TransformerBlock(dim_model[-1], dim_model[-1])
        self.mlp_output = MLP([dim_model[0], 64, oc], norm=None)

    def forward(self, x, pos, batch=None):
        if x is None:
            x = torch.ones((pos.shape[0], 1), device=pos.device)
        ox, op, ob = [], [], []
        x = self.mlp_input(x); ei = knn_graph(pos, k=self.k, batch=batch)
        x = self.transformer_input(x, pos, ei); ox.append(x); op.append(pos); ob.append(batch)
        for i in range(len(self.transformers_down)):
            x, pos, batch = self.transition_down[i](x, pos, batch=batch)
            ei = knn_graph(pos, k=self.k, batch=batch); x = self.transformers_down[i](x, pos, ei)
            ox.append(x); op.append(pos); ob.append(batch)
        x = self.mlp_summit(x); ei = knn_graph(pos, k=self.k, batch=batch)
        x = self.transformer_summit(x, pos, ei)
        for i in range(len(self.transformers_down)):
            x = self.transition_up[-i - 1](x=ox[-i - 2], x_sub=x, pos=op[-i - 2],
                                           pos_sub=op[-i - 1], batch_sub=ob[-i - 1], batch=ob[-i - 2])
            ei = knn_graph(op[-i - 2], k=self.k, batch=ob[-i - 2])
            x = self.transformers_up[-i - 1](x, op[-i - 2], ei)
        return F.log_softmax(self.mlp_output(x), dim=-1)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IN_CH = 4  # features por punto: normal(3) + gris CBCT sintético(1)


def make_model(in_ch):
    """Modelo fresco. in_ch=1 -> pos-only (x=None, geometría); in_ch=4 -> +features."""
    return Net(in_ch, K, dim_model=[32, 64, 128, 256, 512], k=16).to(device)


print(f"Point Transformer seg · {sum(p.numel() for p in make_model(IN_CH).parameters())/1e6:.2f}M params · {device}")

Point Transformer seg · 4.59M params · cuda


### 3 · Entrenamiento — loss ponderada + ablación de features (A2)

Dos arreglos del jefe, juntos:

**#1 · Loss ponderada por clase.** La **encía** (clase 0, ~44% de los puntos) domina el gradiente y el
modelo colapsa prediciéndola. Ponderamos por **frecuencia inversa** (*median-frequency balancing*,
peso topado a 50): un diente pesa ~100× más que la encía, así el modelo no puede «ganar» ignorándolos.

**#2 · Features por punto — ablación (A2).** El modelo *pos-only* (solo geometría) memoriza el train
pero no generaliza el FDI. Añadimos **features por punto** y hacemos una **ablación** con el mismo
split/semilla para **atribuir el mérito**: **`pos-only`** (in_ch=1), **`normales`** (3),
**`gris sintético`** (1, gris CBCT *swap-ready*) y **`normal+gris`** (4). Cada canal se selecciona
recortando columnas de `d.x`. Así se ve, medido, qué feature mueve la generalización y si el gris
aporta señal **propia** o solo replica lo que ya dan las normales.

Diagnóstico: **`tooth_acc`** = acierto SOLO en puntos de diente (no-encía) — el número honesto (el
`mIoU` está inflado por «parte ausente → 1.0»). El `tooth_acc(test)` final se **promedia sobre 5
sorteos** de submuestreo para quitar la varianza de `FixedPoints`.

In [6]:
import time

# --- Pesos por clase (comentario #1 del jefe) --------------------------------
# La ENCÍA (clase 0) domina el gradiente; ponderamos por frecuencia inversa
# (median-frequency balancing): peso_c = mediana(freq)/freq_c, topado a 50.
counts = torch.zeros(K)
for d in train_list:
    counts += torch.bincount(d.y, minlength=K).float()
freq = counts / counts.sum()
present = counts > 0
weights = torch.zeros(K)
weights[present] = (freq[present].median() / freq[present]).clamp(max=50.0)
W = weights.to(device)
ratio = (weights[1:][present[1:]].mean() / weights[0]).item()
print(f"encía (clase 0): {freq[0] * 100:.1f}% de los puntos · peso {weights[0]:.3f} · "
      f"un diente pesa ~{ratio:.0f}x más en la loss")

# --- Ablación A2: ¿de dónde viene la mejora? ---------------------------------
# d.x = [nx, ny, nz, gris]. Seleccionamos columnas por config para ATRIBUIR el
# mérito entre NORMALES (geometría local) y GRIS CBCT sintético.
COLS = {"pos-only (geom)": None, "normales": [0, 1, 2],
        "gris sintético": [3], "normal+gris": [0, 1, 2, 3]}


def train(model, optimizer, cols):
    model.train(); tl = tcorr = ttot = 0
    for data in train_loader:
        data = data.to(device); optimizer.zero_grad()
        x = data.x[:, cols] if cols else None
        out = model(x, data.pos, data.batch)
        loss = F.nll_loss(out, data.y, weight=W)
        loss.backward(); optimizer.step()
        tl += loss.item() * data.num_graphs
        pred = out.argmax(1); tm = data.y != 0        # solo dientes (foreground)
        tcorr += pred[tm].eq(data.y[tm]).sum().item(); ttot += int(tm.sum())
    return tcorr / max(ttot, 1)


@torch.no_grad()
def tooth_acc_test(model, cols, passes=1):
    """acc en puntos de diente. passes>1 promedia sorteos de FixedPoints (el test
    re-submuestrea cada pasada) -> quita el ruido de submuestreo."""
    model.eval(); accs = []
    for _ in range(passes):
        tcorr = ttot = 0
        for data in test_loader:
            data = data.to(device); x = data.x[:, cols] if cols else None
            pred = model(x, data.pos, data.batch).argmax(1)
            tm = data.y != 0
            tcorr += pred[tm].eq(data.y[tm]).sum().item(); ttot += int(tm.sum())
        accs.append(tcorr / max(ttot, 1))
    return sum(accs) / len(accs)


EPOCHS = 20
results = {}
for cfg, cols in COLS.items():
    ic = len(cols) if cols else 1
    torch.manual_seed(0)
    model = make_model(ic)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5)
    t0 = time.perf_counter(); tacc = 0.0
    for epoch in range(1, EPOCHS + 1):
        tacc = train(model, opt, cols)
        sch.step()
    tt = tooth_acc_test(model, cols, passes=5)          # headline denoised
    results[cfg] = (tacc, tt)
    print(f"{cfg:16s} in_ch={ic}  train {tacc:.3f} · test {tt:.3f}  ({time.perf_counter()-t0:.0f}s)")

print("\n=== ABLACIÓN A2 (tooth_acc, mismo split/semilla) ===")
base = results['pos-only (geom)'][1]
for cfg, (tr, te) in results.items():
    print(f"{cfg:16s} train {tr:.3f} · test {te:.3f}   (Δ vs pos-only {te - base:+.3f})")
print(f"\nañadir gris a normales: {results['normal+gris'][1] - results['normales'][1]:+.3f} "
      f"(≈0 -> redundante; el gris sintético NO aporta señal independiente)")

encía (clase 0): 44.0% de los puntos · peso 0.038 · un diente pesa ~147x más en la loss


/home/lgarbayo/.venvs/dental-gpu/lib/python3.13/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


pos-only (geom)  in_ch=1  train 0.779 · test 0.013  (100s)


normales         in_ch=3  train 0.827 · test 0.756  (99s)


gris sintético   in_ch=1  train 0.725 · test 0.738  (98s)


normal+gris      in_ch=4  train 0.832 · test 0.764  (97s)

=== ABLACIÓN A2 (tooth_acc, mismo split/semilla) ===
pos-only (geom)  train 0.779 · test 0.013   (Δ vs pos-only +0.000)
normales         train 0.827 · test 0.756   (Δ vs pos-only +0.744)
gris sintético   train 0.725 · test 0.738   (Δ vs pos-only +0.725)
normal+gris      train 0.832 · test 0.764   (Δ vs pos-only +0.752)

añadir gris a normales: +0.008 (≈0 -> redundante; el gris sintético NO aporta señal independiente)


### 4 · A3 · boundary + centroid loss (estilo IOSNet / DDMF)

A2 dejó `tooth_acc` alto con **normales** (0.84), pero donde más se falla es en el **borde
encía-diente** y entre dientes vecinos. La rama de malla de DDMF (**IOSNet**) ataca justo eso con dos
pérdidas extra, que replicamos sobre la mejor config (**normales**, sin CBCT):

- **boundary loss**: CE extra sobre el **10% de puntos frontera** (los que más vecinos kNN de etiqueta
  distinta tienen) → foco en el borde diente-encía / diente-diente.
- **centroid loss**: por diente, acerca el **centro geométrico predicho** (blando, ponderado por
  probabilidad) al **centro real** → regulariza la compacidad de cada clase.

Comparamos **normales · CE** vs **normales · CE + boundary + centroid** (mismo split/semilla) y, además
del `tooth_acc` global, medimos el **acierto en los puntos frontera** — donde debe verse la mejora.

In [7]:
# A3: boundary loss + centroid loss (estilo IOSNet/DDMF) sobre la mejor config (normales).
COLS_A3 = [0, 1, 2]            # normales
LAMBDA_B, LAMBDA_C = 0.5, 0.3  # pesos de las pérdidas extra


def boundary_mask(pos, y, batch, k=16, frac=0.10):
    """score = fracción de vecinos kNN con etiqueta distinta; frontera = top-frac
    por grafo (borde fino y controlado, estilo top-5% de DDMF)."""
    ei = knn_graph(pos, k=k, batch=batch, loop=False)
    src, dst = ei[0], ei[1]
    diff = (y[src] != y[dst]).float()
    score = torch.zeros(y.size(0), device=y.device)
    score.scatter_add_(0, dst, diff)
    m = torch.zeros(y.size(0), dtype=torch.bool, device=y.device)
    for b in batch.unique():
        sel = (batch == b).nonzero(as_tuple=True)[0]
        m[sel[score[sel].topk(max(1, int(frac * len(sel)))).indices]] = True
    return m


def centroid_loss(logp, pos, y, batch):
    """por grafo y clase presente: centro predicho (blando) vs centro GT."""
    prob = logp.exp(); tot = logp.new_tensor(0.0); n = 0
    for b in batch.unique():
        sel = batch == b; p, yy, pr = pos[sel], y[sel], prob[sel]
        for c in yy.unique().tolist():
            gc = p[yy == c].mean(0); w = pr[:, c]
            pc = (w[:, None] * p).sum(0) / w.sum().clamp_min(1e-6)
            tot = tot + (pc - gc).pow(2).sum().clamp_min(1e-12).sqrt(); n += 1
    return tot / max(n, 1)


def train_a3(model, opt, extra):
    model.train(); tc = tt = 0
    for data in train_loader:
        data = data.to(device); opt.zero_grad()
        logp = model(data.x[:, COLS_A3], data.pos, data.batch)
        loss = F.nll_loss(logp, data.y, weight=W)
        if extra:
            bm = boundary_mask(data.pos, data.y, data.batch)
            if bm.any():
                loss = loss + LAMBDA_B * F.nll_loss(logp[bm], data.y[bm], weight=W)
            loss = loss + LAMBDA_C * centroid_loss(logp, data.pos, data.y, data.batch)
        loss.backward(); opt.step()
        pred = logp.argmax(1); tm = data.y != 0
        tc += pred[tm].eq(data.y[tm]).sum().item(); tt += int(tm.sum())
    return tc / max(tt, 1)


@torch.no_grad()
def eval_a3(model, passes=5):
    """tooth_acc global y en los puntos FRONTERA (diente-encía/diente-diente)."""
    model.eval(); gl = []; bd = []
    for _ in range(passes):
        gc = gt = bc = bt = 0
        for data in test_loader:
            data = data.to(device)
            pred = model(data.x[:, COLS_A3], data.pos, data.batch).argmax(1)
            tm = data.y != 0
            gc += pred[tm].eq(data.y[tm]).sum().item(); gt += int(tm.sum())
            bm = boundary_mask(data.pos, data.y, data.batch) & tm
            bc += pred[bm].eq(data.y[bm]).sum().item(); bt += int(bm.sum())
        gl.append(gc / max(gt, 1)); bd.append(bc / max(bt, 1))
    return sum(gl) / len(gl), sum(bd) / len(bd)


EPOCHS = 20
res_a3 = {}
for name, extra in [("normales · CE (baseline)", False),
                    ("normales · CE+boundary+centroid", True)]:
    torch.manual_seed(0)
    model = make_model(len(COLS_A3))
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5)
    t0 = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        train_a3(model, opt, extra); sch.step()
    ta, tb = eval_a3(model)
    res_a3[name] = (ta, tb)
    print(f"{name:34s} tooth_acc {ta:.3f} · frontera {tb:.3f}  ({time.perf_counter()-t0:.0f}s)")

print("\n=== A3 (mismo split/semilla) ===")
b = res_a3["normales · CE (baseline)"]; a = res_a3["normales · CE+boundary+centroid"]
print(f"Δ tooth_acc global : {a[0]-b[0]:+.3f}")
print(f"Δ tooth_acc frontera: {a[1]-b[1]:+.3f}  (donde las pérdidas extra deben ayudar)")

normales · CE (baseline)           tooth_acc 0.758 · frontera 0.657  (99s)


normales · CE+boundary+centroid    tooth_acc 0.792 · frontera 0.707  (172s)

=== A3 (mismo split/semilla) ===
Δ tooth_acc global : +0.034
Δ tooth_acc frontera: +0.051  (donde las pérdidas extra deben ayudar)


### 5 · Resumen

**Experimento.** Segmentación FDI **por punto** sobre **Teeth3DS+ completo** (prototipo del
`segmentation-agent`). Point Transformer · 600 mallas / 300 pacientes · 32 clases · split por paciente
240/60 (seed 42) · loss ponderada por clase · diagnóstico `tooth_acc` (media de 5 sorteos).

**A2 · Ablación de features** (mismo split/semilla):

| config | in_ch | train | **test** |
|---|---|---|---|
| pos-only (geometría) | 1 | 0.78 | **0.01** |
| normales | 3 | 0.83 | **0.76** |
| gris CBCT sintético | 1 | 0.73 | **0.74** |
| normal + gris | 4 | 0.83 | **0.76** |

- **`pos-only` no generaliza** (test ~0.01–0.08 según ejecución): se apoya solo en la **posición
  absoluta**, que no transfiere entre pacientes.
- **Cualquier descriptor LOCAL por punto lo arregla** → salto a **~0.74–0.84**.
- Las tres configs con features son **estadísticamente indistinguibles** y el gris **no aporta** sobre
  las normales → **redundantes**. Se adopta **normales** (la más simple).

**A3 · boundary + centroid loss** (estilo IOSNet/DDMF, sobre normales, **sin CBCT**):

| | `tooth_acc` | **frontera** |
|---|---|---|
| normales · CE | 0.758 | 0.657 |
| + boundary + centroid | 0.792 | 0.707 |
| **Δ** | **+0.034** | **+0.051** |

- Mejora **modesta pero consistente** y **mayor en la frontera** (+0.05) que global (+0.03) — justo
  donde se diseñó (borde diente-encía).

**Conclusiones.**
1. **La palanca de generalización es un descriptor geométrico LOCAL por punto** (normales). No es «más
   datos» ni «el CBCT».
2. **El gris CBCT *sintético* no valida el CBCT real** (es geometría redundante). Su valor llega por
   **FUSIÓN**, no por vértice: **DDMF** ([arXiv:2203.05784](https://arxiv.org/abs/2203.05784), 503
   pacientes CBCT+IOS) registra CBCT↔malla (RANSAC-FPFH + ICP multiescala → 0.17 mm) y segmenta la
   malla (IOSNet) con **CE + boundary + centroid**, sin intensidad.
3. **Las pérdidas boundary/centroid mejoran el borde con la encía** usando solo Teeth3DS+.

**Caveats.** Números con **varianza ±~0.05–0.1 entre ejecuciones** (no-determinismo GPU + submuestreo
aleatorio) — lo robusto es lo **cualitativo**, no el 2º decimal. `tooth_acc(test)` = media de 5
sorteos. `mIoU` no informativo (parte-ausente → 1.0).

**Conexión / siguiente.** Esqueleto del **`segmentation-agent`** (nube → FDI por punto). El canal de
gris queda **swap-ready** para CBCT real; el `cbct-agent` irá por **segmentación propia + registro**
(fusión), no por gris-por-vértice. Validar el FDI contra el informe = `fdi-consistency-agent` (ADR 003).